In [ ]:
# Run once if packages are missing
#!pip install -r requirements.txt

  Obtaining dependency information for sentence-transformers from https://files.pythonhosted.org/packages/e2/9c/2fa7224058cad8df68d84bafee21716f30892cecc7ad1ad73bde61d23754/sentence_transformers-5.3.0-py3-none-any.whl.metadata
  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/6e/01/624c4324ca01f66ae4c7cd1b74eb16fb52596dce66dbe51eff95ef9e7a4c/torch-2.10.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for transformers<6.0.0,>=4.41.0 from https://files.pythonhosted.org/packages/b8/88/ae8320064e32679a5429a2c9ebbc05c2bf32cefb6e076f9b07f6d685a9b4/transformers-5.3.0-py3-none-any.whl.metadata
  Obtaining dependency information for huggingface-hub>=0.20.0 from https://files.pythonhosted.org/packages/92/e3/e3a44f54c8e2f28983fcf07f13d4260b37bd6a0d3a081041bc60b91d230e/huggingface_hub-1.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/9f/c4/0ab22726a04ede56f6


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ==============================
# 1. Imports
# ==============================

import os
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import AzureOpenAI
from dotenv import load_dotenv

In [ ]:
# ==============================
# 2. Load environment variables
# ==============================

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
# embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")




modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\Raivis.Skadins\git\simple-rag-demo\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Raivis.Skadins\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# ==============================
# 3. Load knowledge base
# ==============================

with open("./data/knowledge.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Each paragraph becomes one chunk
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print("Chunks:")
for c in chunks:
    print("-", c)


Chunks:
- The Eiffel Tower is located in Paris, France. It was completed in 1889 and is one of the most recognizable landmarks in the world.
- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.
- The Amazon rainforest is the largest tropical rainforest in the world. It spans multiple countries in South America and contains immense biodiversity.
- The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface.


In [ ]:
# ==============================
# 4. Create embeddings
# ==============================

# def get_embedding(text):
#     response = client.embeddings.create(
#         model=embedding_deployment,
#         input=text
#     )
#     return response.data[0].embedding
#chunk_embeddings = [get_embedding(chunk) for chunk in chunks]
#embedding_dim = len(chunk_embeddings[0])


def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

#index = faiss.IndexFlatL2(embedding_dim)
#index.add(np.array(chunk_embeddings).astype("float32"))

index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)


APIConnectionError: Connection error.

In [ ]:
# ==============================
# 6. User question
# ==============================

question = "Which ocean is the largest on Earth?"

print("Question:", question)


# ==============================
# 7. Embed the question
# ==============================

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")



In [ ]:
# ==============================
# 8. Retrieve top 2 chunks
# ==============================

k = 2

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)


In [ ]:
# ==============================
# 9. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using ONLY the provided context.

Context:
{context}

Question:
{question}

Answer:
"""


In [ ]:


# ==============================
# 10. Call GPT-4o
# ==============================

response = client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)